# Reinforcement Learning: On-Policy vs Off-Policy Comparison

This notebook demonstrates the difference between **on-policy** and **off-policy** RL methods using a grid world environment.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import random
from IPython.display import HTML
import matplotlib.animation as animation

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## Grid World Environment

We create a grid world with:
- **Start state**: Top-left corner
- **Goal state**: Bottom-right corner (the prize!)
- **Obstacles**: Walls that the agent cannot pass through
- **Rewards**: Reaching the goal gives +10, each step costs -0.1

In [ ]:
class GridWorld:
    def __init__(self, width=8, height=8, seed=42):
        self.width = width
        self.height = height
        self.n_states = width * height
        self.n_actions = 4  # up, down, left, right
        self.actions = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}  # up, down, left, right
        
        np.random.seed(seed)
        self.start = (0, 0)
        self.goal = (height - 1, width - 1)
        
        # Create obstacles (random walls)
        self.obstacles = set()
        n_obstacles = 10
        while len(self.obstacles) < n_obstacles:
            obs = (np.random.randint(0, height), np.random.randint(0, width))
            if obs != self.start and obs != self.goal:
                self.obstacles.add(obs)
        
        self.obstacles = frozenset(self.obstacles)
        self.step_count = 0
        self.max_steps = 100
    
    def reset(self):
        self.current_state = self.start
        self.step_count = 0
        return self._state_to_idx(self.current_state)
    
    def step(self, action):
        self.step_count += 1
        
        if self.step_count >= self.max_steps:
            return self._state_to_idx(self.current_state), -1, True, {}
        
        dx, dy = self.actions[action]
        new_row = self.current_state[0] + dx
        new_col = self.current_state[1] + dy
        
        # Check boundaries
        if 0 <= new_row < self.height and 0 <= new_col < self.width:
            new_state = (new_row, new_col)
            
            # Check obstacles
            if new_state in self.obstacles:
                return self._state_to_idx(self.current_state), -0.5, False, {}
            
            self.current_state = new_state
            
            # Check goal
            if new_state == self.goal:
                return self._state_to_idx(new_state), 10, True, {}
            
            return self._state_to_idx(new_state), -0.1, False, {}
        
        return self._state_to_idx(self.current_state), -0.1, False, {}
    
    def _state_to_idx(self, state):
        return state[0] * self.width + state[1]
    
    def idx_to_state(self, idx):
        return (idx // self.width, idx % self.width)
    
    def render(self, agent_pos=None, title=None):
        fig, ax = plt.subplots(1, 1, figsize=(6, 6))
        
        for row in range(self.height):
            for col in range(self.width):
                state = (row, col)
                if state == self.start:
                    color = 'lightgreen'
                    marker = 'S'
                elif state == self.goal:
                    color = 'gold'
                    marker = 'G'
                elif state in self.obstacles:
                    color = 'black'
                    marker = ''
                elif agent_pos == state:
                    color = 'dodgerblue'
                    marker = 'A'
                else:
                    color = 'white'
                    marker = ''
                
                ax.add_patch(plt.Rectangle((col, self.height - row - 1), 1, 1, 
                                           facecolor=color, edgecolor='gray', linewidth=1))
                if marker:
                    ax.text(col + 0.5, self.height - row - 0.5, marker, 
                           ha='center', va='center', fontsize=16, fontweight='bold')
        
        ax.set_xlim(-0.1, self.width + 0.1)
        ax.set_ylim(-0.1, self.height + 0.1)
        ax.set_aspect('equal')
        ax.axis('off')
        
        if title:
            ax.set_title(title, fontsize=14)
        
        plt.tight_layout()
        return fig

## Epsilon-Greedy Policy

In [ ]:
def epsilon_greedy(q_table, state, n_actions, epsilon):
    if random.random() < epsilon:
        return random.randint(0, n_actions - 1)
    return np.argmax(q_table[state])

## On-Policy: SARSA (State-Action-Reward-State-Action)

SARSA updates its Q-values using the action that will actually be taken in the next state.
This means it learns about the policy it's following.

**Update rule:**
```
Q(s,a) ← Q(s,a) + α[r + γQ(s',a') - Q(s,a)]
```

Where `a'` is the action selected by the current policy (ε-greedy).

In [ ]:
def sarsa(env, n_episodes=500, alpha=0.1, gamma=0.99, epsilon_start=1.0, epsilon_end=0.01):
    q_table = np.zeros((env.n_states, env.n_actions))
    rewards_per_episode = []
    
    for episode in range(n_episodes):
        state = env.reset()
        action = epsilon_greedy(q_table, state, env.n_actions, epsilon_start)
        total_reward = 0
        done = False
        
        while not done:
            next_state, reward, done, _ = env.step(action)
            next_action = epsilon_greedy(q_table, next_state, env.n_actions, 
                                        epsilon_start * (epsilon_end / epsilon_start) ** (episode / n_episodes))
            
            # SARSA update
            q_table[state, action] += alpha * (reward + gamma * q_table[next_state, next_action] - q_table[state, action])
            
            state = next_state
            action = next_action
            total_reward += reward
        
        rewards_per_episode.append(total_reward)
    
    return q_table, rewards_per_episode

## Off-Policy: Q-Learning

Q-Learning updates its Q-values using the maximum Q-value over all possible next actions.
This means it learns about the optimal policy even while following a different (exploratory) policy.

**Update rule:**
```
Q(s,a) ← Q(s,a) + α[r + γmax_a' Q(s',a') - Q(s,a)]
```

Where `max_a'` is the action with the highest Q-value (greedy), regardless of the current policy.

In [ ]:
def q_learning(env, n_episodes=500, alpha=0.1, gamma=0.99, epsilon_start=1.0, epsilon_end=0.01):
    q_table = np.zeros((env.n_states, env.n_actions))
    rewards_per_episode = []
    
    for episode in range(n_episodes):
        state = env.reset()
        epsilon = epsilon_start * (epsilon_end / epsilon_start) ** (episode / n_episodes)
        total_reward = 0
        done = False
        
        while not done:
            action = epsilon_greedy(q_table, state, env.n_actions, epsilon)
            next_state, reward, done, _ = env.step(action)
            
            # Q-Learning update
            q_table[state, action] += alpha * (reward + gamma * np.max(q_table[next_state]) - q_table[state, action])
            
            state = next_state
            total_reward += reward
        
        rewards_per_episode.append(total_reward)
    
    return q_table, rewards_per_episode

## Visualize the Environment

In [ ]:
env = GridWorld(width=8, height=8)
print(f"Grid Size: {env.width}x{env.height}")
print(f"Number of States: {env.n_states}")
print(f"Start: {env.start}")
print(f"Goal: {env.goal}")
print(f"Number of Obstacles: {len(env.obstacles)}")

env.render(title='Grid World Environment')

## Train Both Methods

In [ ]:
n_episodes = 1000
print("Training SARSA (On-Policy)...")
sarsa_q, sarsa_rewards = sarsa(env, n_episodes=n_episodes)
print("Training Q-Learning (Off-Policy)...")
qlearn_q, qlearn_rewards = q_learning(env, n_episodes=n_episodes)
print("Training complete!")

## Plot Training Performance

In [ ]:
def smooth(data, weight=0.6):
    smoothed = []
    last = data[0]
    for point in data:
        smoothed_val = last * weight + (1 - weight) * point
        smoothed.append(smoothed_val)
        last = smoothed_val
    return smoothed

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw rewards
axes[0].plot(sarsa_rewards, alpha=0.4, color='blue', label='SARSA')
axes[0].plot(qlearn_rewards, alpha=0.4, color='red', label='Q-Learning')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Training Rewards (Raw)')
axes[0].legend()

# Smoothed rewards
axes[1].plot(smooth(sarsa_rewards), linewidth=2, color='blue', label='SARSA')
axes[1].plot(smooth(qlearn_rewards), linewidth=2, color='red', label='Q-Learning')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Total Reward')
axes[1].set_title('Training Rewards (Smoothed)')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_rewards.png', dpi=150, bbox_inches='tight')
plt.show()

## Compare Final Performance

In [ ]:
def evaluate_policy(env, q_table, n_eval=100, max_steps=100):
    successes = 0
    avg_steps = []
    
    for _ in range(n_eval):
        state = env.reset()
        done = False
        steps = 0
        
        while not done and steps < max_steps:
            action = np.argmax(q_table[state])
            state, reward, done, _ = env.step(action)
            steps += 1
        
        if done and reward == 10:
            successes += 1
            avg_steps.append(steps)
    
    return successes / n_eval * 100, np.mean(avg_steps) if avg_steps else float('inf')

sarsa_success, sarsa_avg_steps = evaluate_policy(env, sarsa_q)
qlearn_success, qlearn_avg_steps = evaluate_policy(env, qlearn_q)

print(f"{'Method':<15} {'Success Rate':<15} {'Avg Steps to Goal'}")
print("-" * 50)
print(f"{'SARSA':<15} {sarsa_success:<15.1f} {sarsa_avg_steps:.1f}")
print(f"{'Q-Learning':<15} {qlearn_success:<15.1f} {qlearn_avg_steps:.1f}")

## Visualize Learned Policies

In [ ]:
def visualize_policy(env, q_table, title):
    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    
    arrows = ['↑', '↓', '←', '→']
    
    for row in range(env.height):
        for col in range(env.width):
            state = (row, col)
            idx = env._state_to_idx(state)
            
            if state == env.start:
                color = 'lightgreen'
                text = 'S'
            elif state == env.goal:
                color = 'gold'
                text = 'G'
            elif state in env.obstacles:
                color = 'black'
                text = ''
            else:
                color = 'white'
                best_action = np.argmax(q_table[idx])
                text = arrows[best_action]
            
            ax.add_patch(plt.Rectangle((col, env.height - row - 1), 1, 1, 
                                       facecolor=color, edgecolor='gray', linewidth=1))
            ax.text(col + 0.5, env.height - row - 0.5, text, 
                   ha='center', va='center', fontsize=12, fontweight='bold')
    
    ax.set_xlim(-0.1, env.width + 0.1)
    ax.set_ylim(-0.1, env.height + 0.1)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=14)
    
    plt.tight_layout()
    return fig

fig1 = visualize_policy(env, sarsa_q, 'SARSA Policy (On-Policy)')
fig2 = visualize_policy(env, qlearn_q, 'Q-Learning Policy (Off-Policy)')
plt.show()

## Run an Episode with Learned Policy

In [ ]:
def run_episode_greedy(env, q_table, max_steps=100):
    frames = []
    state = env.reset()
    
    for _ in range(max_steps):
        frames.append(env.current_state)
        action = np.argmax(q_table[state])
        state, reward, done, _ = env.step(action)
        if done:
            frames.append(env.current_state)
            break
    
    return frames

def create_animation(env, frames, title):
    fig, ax = plt.subplots(figsize=(6, 6))
    
    def init():
        ax.clear()
        return []
    
    def animate(i):
        ax.clear()
        pos = frames[i] if i < len(frames) else frames[-1]
        
        for row in range(env.height):
            for col in range(env.width):
                state = (row, col)
                if state == env.start:
                    color = 'lightgreen'
                elif state == env.goal:
                    color = 'gold'
                elif state in env.obstacles:
                    color = 'black'
                elif state == pos:
                    color = 'dodgerblue'
                else:
                    color = 'white'
                
                ax.add_patch(plt.Rectangle((col, env.height - row - 1), 1, 1, 
                                           facecolor=color, edgecolor='gray', linewidth=1))
        
        ax.set_xlim(-0.1, env.width + 0.1)
        ax.set_ylim(-0.1, env.height + 0.1)
        ax.set_aspect('equal')
        ax.axis('off')
        ax.set_title(f'{title} - Step {i+1}/{len(frames)}', fontsize=12)
        return []
    
    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=len(frames), interval=500, blit=True)
    return anim

print("Running SARSA episode...")
sarsa_frames = run_episode_greedy(env, sarsa_q)
print(f"Steps: {len(sarsa_frames)}")

print("Running Q-Learning episode...")
qlearn_frames = run_episode_greedy(env, qlearn_q)
print(f"Steps: {len(qlearn_frames)}")

## Summary: On-Policy vs Off-Policy

### Key Differences:

| Aspect | On-Policy (SARSA) | Off-Policy (Q-Learning) |
|--------|-------------------|-------------------------|
| **Learns** | Policy it follows | Optimal policy |
| **Update** | Uses actual next action | Uses max Q(s',a') |
| **Exploration** | Committed to exploration during learning | Can explore while learning optimal |
| **Risk** | Safer, avoids dangerous actions | Can be riskier, learns about risky paths |
| **Convergence** | Converges to good policy | Converges to optimal policy |

### In this example:
- **SARSA** tends to find safer, more conservative paths
- **Q-Learning** finds shorter but potentially riskier paths

The choice depends on your application:
- Use **SARSA** when safety is critical (robotics near humans)
- Use **Q-Learning** when you want optimal performance